In [1]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

# Astro
from astropy.table import Table 
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
from astropy.nddata import Cutout2D
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.wcs.utils import proj_plane_pixel_scales

# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [2]:
# Before processing
# RI: /data0/Postech/data/injected_psf_kmtnet_20230925/csv/transient_meta_table.csv
# NGI: /data0/PSF_INJECTION_NGI/real.csv , /data0/PSF_INJECTION_NGI/bogus.csv

# After processing
# - RI: /home/postech/projects/kmtnet/data/meta_random_injection.csv
# - NGI: /home/postech/projects/kmtnet/data/meta_near_galaxy_injection-v1.0.csv
# - Test sample: /data0/Postech/data/test/meta.csv

In [3]:
ri_table_path = '../data/meta_random_injection.csv'
ngi_table_path = '../data/meta_near_galaxy_injection-v1.0.csv'
test_table_path = '../data/meta.csv'

In [4]:
ritbl = Table.read(ri_table_path)
ngitbl = Table.read(ngi_table_path)
testtbl = Table.read(test_table_path)

id,obs,band,mag_injected,mag_measured,snr,fwhm,ellipticity,ra,dec,x_image,y_image,class,prob,label,refim,newim,subim,temim
str77,str4,str1,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,float64,int64,str86,str86,str86,str86
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000079,CTIO,R,18.61504906677617,18.616300000000003,58.13953488372093,2.03651568,0.106,118.18322667185748,-61.02300467056323,2045.0568,1652.9672,real,1.0,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000079.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000079.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000079.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000079.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000082,CTIO,R,20.54184938834267,20.6937,11.976047904191615,4.096764,0.361,118.09255350116185,-61.02546349077948,2441.051,1643.7554,real,0.9988507032394408,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000082.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000082.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000082.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000082.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000188,CTIO,R,17.012955593406495,17.0156,227.27272727272725,1.9506744,0.088,117.4309266902172,-61.02156735350901,5325.064,1749.9973,real,1.0,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000188.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000188.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000188.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000188.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000194,CTIO,R,20.073591863949087,20.1809,16.39344262295082,2.9628234,0.099,117.93456464854545,-61.0181112151726,3128.0676,1729.3192,real,0.999998927116394,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000194.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000194.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000194.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000194.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000210,CTIO,R,16.933924629035076,16.913600000000002,243.90243902439025,1.96964424,0.089,117.41131751676988,-61.01864424681149,5409.9668,1778.0396,real,1.0,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000210.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000210.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000210.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000210.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000211,CTIO,R,20.820579580477347,20.652500000000003,9.416195856873824,2.41417908,0.34,117.40757150919735,-61.02245679268022,5426.9082,1743.6078,real,0.9969242215156556,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000211.ref.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000211.new.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000211.sub.fits,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000211.tem.fits
hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000272,CTIO,R,16.348620493633533,16.3313,400.00000000000006,1.95357204,0.113,118.11445375129176,-61.00503254123779,2340.033,1823.9822,real,1.0,1,hdCalib.KMTNet_CTIO.S230518h_2686.20230519-003329.R.360.stack.trim_0x0.000272